In [1]:
"""
Reservoir Fluid Properties (Oil)
----------------------------------------
Formula reference: Tarek Ahmad - Reservoir Engineering Handbook
Note: Formula is in public domain. This implementation is original code.

Author: Wan Muhammad Faiz zaki
License: MIT
"""

'\nReservoir Fluid Properties (Oil)\n----------------------------------------\nFormula reference: Tarek Ahmad - Reservoir Engineering Handbook\nNote: Formula is in public domain. This implementation is original code.\n\nAuthor: Wan Muhammad Faiz zaki\nLicense: MIT\n'

PROPERTIES OF CRUDE OIL SYSTEMS

• Fluid gravity

• Specific gravity of the solution gas

• Gas solubility

• Bubble-point pressure

• Oil formation volume factor

• Isothermal compressibility coefficient of
undersaturated crude oils

• Oil density

• Total formation volume factor

• Crude oil viscosity

• Surface tension

**Crude Oil Gravity**

The specific gravity of a crude oil is defined as the ratio of the density of the oil to that of water.

In [ ]:
def oil_gravity(rho_oil):
  rho_water=62.4
  og = rho_oil/rho_water
  return og

def API_gravity(og):
  API= (141.5/og)-131.5
  return API
#The API gravities of crude oils usually range from 47° API for the lighter crude oils to 10° API for the heavier asphaltic crude oils.


**Example**


Calculate the specific gravity and the API gravity of a crude oil system with a measured density of 53 lb/ft3 at standard conditions.

In [ ]:
oil_g = round(oil_gravity(53),3)
print(f"Specifc gravity: {oil_g}")
API_g = round(API_gravity(oil_g),1)
print(f"API gravity: {API_g}° API ")

Specifc gravity: 0.849
API gravity: 35.2° API 


**Specific Gravity of the Solution Gas**

The specific gravity of the solution gas is described by the weighted average of the specific gravities of the separated gas from each separator.

In [ ]:
def gas_specific_gravity(R_sep, gamma_sep, R_st, gamma_st):
    """
    Calculate solution gas specific gravity (gamma_g)

    Parameters:
    R_sep (list): separator gas-oil ratios [scf/STB]
    gamma_sep (list): separator gas specific gravities
    R_st (float): stock tank gas-oil ratio [scf/STB]
    gamma_st (float): stock tank gas specific gravity

    Returns:
    float: solution gas specific gravity
    """

    # numerator: sum(R_sep_i * gamma_sep_i) + R_st * gamma_st
    numerator = sum(R_sep[i] * gamma_sep[i] for i in range(len(R_sep))) + R_st * gamma_st

    # denominator: sum(R_sep_i) + R_st
    denominator = sum(R_sep) + R_st

    gamma_g = numerator / denominator
    return gamma_g

In [ ]:
R_sep = [724,202]
gamma_sep = [0.743, 0.956]
R_st = 58
gamma_st = 1.296

gamma_g = round(gas_specific_gravity(R_sep, gamma_sep, R_st, gamma_st),3)
print(f"Solution gas specific gravity: {gamma_g}")

Solution gas specific gravity: 0.819


**Gas Solubility**

The gas solubility Rs is defined as the number of standard cubic feet of gas which will dissolve in one stock-tank barrel of crude oil at certain pressure and temperature.


 Five empirical correlations for estimating the gas solubility

 • Standing’s correlation

• The Vasquez-Beggs correlation

• Glaso’s correlation

• Marhoun’s correlation

• The Petrosky-Farshad correlation

**Standing’s correlation**

In [ ]:
def standing_rs(Pb, T, API, sg):
  x = round(0.0125*API - 0.00091*(T - 460),3)
  y = round(10**x,3)
  Rs = round(sg*(((Pb/18.2)+1.4)*y)**1.2048,0)
  return Rs

In [ ]:
API=47.1
T=250+460
sg=0.851
Pb=2377

Rs = standing_rs(Pb,T,API,sg)
print(f"Standing's correlation: {Rs}")

Standing's correlation: 831.0


**Example**

The following experimental PVT data on six different crude oil items are available. Results are based on two-stage surface separation.

In [ ]:
import pandas as pd

data = {
    "Oil #": [1, 2, 3, 4, 5, 6],
    "T": [250+460, 220+460, 260+460, 237+460, 218+460, 180+460],
    "Pb": [2377+14.7, 2620+14.7, 2051+14.7, 2884+14.7, 3045+14.7, 4239+14.7],
    "Rs (scf/STB)": [751, 768, 693, 968, 943, 807],
    "Bo (bbl/STB)": [1.528, 1.474, 1.529, 1.619, 1.570, 1.385],
    "ρo": [38.13, 40.95, 37.37, 38.92, 37.70, 46.79],
    "co (1/psi)": [22.14e-6, 18.75e-6, 22.69e-6, 21.51e-6, 24.16e-6, 11.45e-6],
    "P (psig)": [2689, 2810, 2526, 2942, 3273, 4370],
    "Psep": [150+14.7, 100+14.7, 100+14.7, 60+14.7, 200+14.7, 85+14.7],
    "Tsep": [60+460, 75+460, 72+460, 120+460, 60+460, 173+460],
    "API": [47.1, 40.7, 48.6, 40.5, 44.2, 27.3],
    "sg": [0.851, 0.855, 0.911, 0.898, 0.781, 0.848]
}

df = pd.DataFrame(data)

df

,Oil #,T,Pb,Rs (scf/STB),Bo (bbl/STB),ρo,co (1/psi),P (psig),Psep,Tsep,API,sg
0,1,710,2391.7,751,1.528,38.13,0.000022,2689,164.7,520,47.1,0.851
1,2,680,2634.7,768,1.474,40.95,0.000019,2810,114.7,535,40.7,0.855
2,3,720,2065.7,693,1.529,37.37,0.000023,2526,114.7,532,48.6,0.911
3,4,697,2898.7,968,1.619,38.92,0.000022,2942,74.7,580,40.5,0.898
4,5,678,3059.7,943,1.570,37.70,0.000024,3273,214.7,520,44.2,0.781
5,6,640,4253.7,807,1.385,46.79,0.000011,4370,99.7,633,27.3,0.848


Using Standing’s correlation, estimate the gas solubility at the bubblepoint pressure and compare with the experimental value in terms of the absolute average error (AAE).

In [ ]:
df1 = df[['T', 'Pb', 'sg', 'API','Rs (scf/STB)']].copy()
df1["X"] = round(0.0125*df1['API'] - 0.00091*(df1['T'] - 460),3)
df1["Y"] = round(10**df1["X"],3)
df1['Rs_Standing'] = df1.apply(lambda row: standing_rs(row['Pb'], row['T'], row['API'], row['sg']), axis=1)
df1["Error %"] = round(abs((df1["Rs (scf/STB)"] - df1["Rs_Standing"])/df1["Rs (scf/STB)"]*100),2)
display(df1.head(6))

,T,Pb,sg,API,Rs (scf/STB),X,Y,Rs_Standing,Error %
0,710,2391.7,0.851,47.1,751,0.361,2.296,837.0,11.45
1,680,2634.7,0.855,40.7,768,0.309,2.037,817.0,6.38
2,720,2065.7,0.911,48.6,693,0.371,2.350,774.0,11.69
3,697,2898.7,0.898,40.5,968,0.291,1.954,915.0,5.48
4,678,3059.7,0.781,44.2,943,0.354,2.259,1011.0,7.21
5,640,4253.7,0.848,27.3,807,0.177,1.503,997.0,23.54


In [ ]:
average_error_percentage = df1['Error %'].mean()
print(f" Absolute Average Error: {average_error_percentage:.2f}%")

 Absolute Average Error: 10.96%


**The Vasquez-Beggs Correlation**

In [ ]:
from numpy import log10, exp
def Rs_vasquez(sg,API,Tsep,Psep,Pb,T):
  sg_s= sg*(1+5.912*(10**-5)*API*(Tsep-460)*log10(Psep/114.7))
  if API > 30:
    C1=0.0178
    C2=1.1870
    C3=23.931
  else:
    C1=0.0362
    C2=1.0937
    C3=25.7240

  Rs= round(C1 * sg_s* (Pb**C2) * exp(C3*(API/T)),0)
  return Rs

In [ ]:
sg=0.851
API=47.1
Tsep=60+460
Psep=150+14.7
Pb=2377+14.7
T=250+460
sg_s= sg*(1+5.912*(10**-5)*API*(Tsep-460)*log10(Psep/114.7))
Rs = Rs_vasquez(sg,API,Tsep,Psep,Pb,T)
print(sg_s)
print(Rs)

0.8733406474157185
779.0


In [ ]:
from numpy import log10
df2 = df[['T', 'Pb', 'sg', 'API','Tsep','Psep','Rs (scf/STB)']].copy()
df2["sg_s"] = df2.apply(lambda row: round(row['sg']*(1+5.912*(10**-5)*row['API']*(row['Tsep']-460)*log10(row['Psep']/114.7)), 3), axis=1)
df2["Rs_Vasquez"] = df2.apply(lambda row: round(abs(Rs_vasquez(row['sg'], row['API'], row['Tsep'], row['Psep'], row['Pb'], row['T'])), 0), axis=1)
df2["Error %"] = round(abs((df2["Rs (scf/STB)"] - df2["Rs_Vasquez"])/df2["Rs (scf/STB)"]*100),2)
df2

,T,Pb,sg,API,Tsep,Psep,Rs (scf/STB),sg_s,Rs_Vasquez,Error %
0,710,2391.7,0.851,47.1,520,164.7,751,0.873,779.0,3.73
1,680,2634.7,0.855,40.7,535,114.7,768,0.855,733.0,4.56
2,720,2065.7,0.911,48.6,532,114.7,693,0.911,702.0,1.30
3,697,2898.7,0.898,40.5,580,74.7,968,0.850,782.0,19.21
4,678,3059.7,0.781,44.2,520,214.7,943,0.814,947.0,0.42
5,640,4253.7,0.848,27.3,633,99.7,807,0.834,841.0,4.21


In [ ]:
average_error_percentage = df2['Error %'].mean()
print(f" Absolute Average Error: {average_error_percentage:.2f}%")

 Absolute Average Error: 5.57%


**Glaso’s Correlation**

In [ ]:
def Rs_glaso(Pb, T, API, sg):
  x = round(2.8869 - (14.1811-3.3093*log10(Pb))**0.5,3)
  y = round(10**x,3)
  Rs= round(sg*((API**0.989 / (T-460)**0.172)*(y))**1.2255,0)
  return Rs


In [ ]:
sg=0.851
API=47.1
Tsep=60+460
Psep=150+14.7
Pb=2377+14.7
T=250+460

x = round(2.8869 - (14.1811-3.3093*log10(Pb))**0.5,3)
y = round(10**x,3)
Rs= (sg*((API**0.989 / (T-460)**0.172)*(y))**1.2255,0)
print(x)
print(y)
print(Rs)

1.155
14.289
(np.float64(737.337107751486), 0)


In [ ]:
df3 = df[['T', 'Pb', 'sg', 'API','Rs (scf/STB)']].copy()
df3["x"] = round(2.8869 - (14.1811-3.3093*log10(df3['Pb']))**0.5,3)
df3["y"] = round(10**df3["x"],3)
df3["Rs_Glaso"] = df3.apply(lambda row: round(Rs_glaso(row['Pb'], row['T'], row['API'], row['sg']), 0), axis=1)
df3["Error %"] = round(abs((df3["Rs (scf/STB)"] - df3["Rs_Glaso"])/df3["Rs (scf/STB)"]*100),2)
df3

,T,Pb,sg,API,Rs (scf/STB),x,y,Rs_Glaso,Error %
0,710,2391.7,0.851,47.1,751,1.155,14.289,737.0,1.86
1,680,2634.7,0.855,40.7,768,1.195,15.668,714.0,7.03
2,720,2065.7,0.911,48.6,693,1.095,12.445,686.0,1.01
3,697,2898.7,0.898,40.5,968,1.237,17.258,826.0,14.67
4,678,3059.7,0.781,44.2,943,1.260,18.197,867.0,8.06
5,640,4253.7,0.848,27.3,807,1.413,25.882,842.0,4.34


In [ ]:
average_error_percentage = df3['Error %'].mean()
print(f" Absolute Average Error: {average_error_percentage:.2f}%")

 Absolute Average Error: 6.16%


**Marhoun’s Correlation**

In [ ]:
def Rs_marhoun(sg,T,Pb,sgo):
  a = 185.843208
  b = 1.877840
  c = -3.1437
  d = -1.32657
  e =  1.398441
  Rs = round((a * (sg**b) * (sgo**c) * (T**d) * Pb)**e,0)
  return Rs


In [ ]:
sg=0.851
API=47.1
Tsep=60+460
Psep=150+14.7
Pb=2377+14.7
T=250+460
sgo = 0.7923
Rs = Rs_marhoun(sg,T,Pb,sgo)
print(Rs)
print(sgo)

740.0
0.7923


In [ ]:
df4 = df[['T', 'Pb', 'sg', 'API','Rs (scf/STB)']].copy()
df4["sgo"] = round(141.5 / (df4["API"] + 131.5),4)
df4["Rs_Marhoun"] = df4.apply(lambda row: round(Rs_marhoun(row['sg'], row['T'], row['Pb'], row['sgo']), 0), axis=1)
df4["Error %"] = round(abs((df4["Rs (scf/STB)"] - df4["Rs_Marhoun"])/df4["Rs (scf/STB)"]*100),2)
df4

,T,Pb,sg,API,Rs (scf/STB),sgo,Rs_Marhoun,Error %
0,710,2391.7,0.851,47.1,751,0.7923,740.0,1.46
1,680,2634.7,0.855,40.7,768,0.8217,792.0,3.12
2,720,2065.7,0.911,48.6,693,0.7857,729.0,5.19
3,697,2898.7,0.898,40.5,968,0.8227,978.0,1.03
4,678,3059.7,0.781,44.2,943,0.8054,845.0,10.39
5,640,4253.7,0.848,27.3,807,0.8911,1186.0,46.96


In [ ]:
average_error_percentage = df4['Error %'].mean()
print(f" Absolute Average Error: {average_error_percentage:.2f}%")

 Absolute Average Error: 11.36%


**The Petrosky-Farshad Correlation**

In [ ]:
def Rs_petrosky(sg,API,Pb,T):
  x = round((7.916*(10**-4) * (API**1.5410)) - (4.561*(10**-5)*(T-460)**1.3911),4)
  y = (10**x)
  Rs = round((((Pb/112.727)+12.340)* sg**0.8439 * y)**1.73184,0)
  return Rs


In [ ]:
sg=0.851
API=47.1
Tsep=60+460
Psep=150+14.7
Pb=2377+14.7
T=250+460
x = round((7.916*(10**-4) * (API**1.5410)) - (4.561*(10**-5)*(T-460)**1.3911),4)
Rs = Rs_petrosky(sg,API,Pb,T)
print(x)
print(Rs)

0.2008
772.0


In [ ]:
df5 = df[['T', 'Pb', 'sg', 'API','Rs (scf/STB)']].copy()
df5["x"] = round((7.916*(10**-4) * (df5["API"]**1.5410)) - (4.561*(10**-5)*(df5["T"]-460)**1.3911),4)
df5["Rs_Petrosky"] = df5.apply(lambda row: round(Rs_petrosky(row['sg'], row['API'], row['Pb'], row['T']), 0), axis=1)
df5["Error %"] = round(abs((df5["Rs (scf/STB)"] - df5["Rs_Petrosky"])/df5["Rs (scf/STB)"]*100),2)
display(df5)

,T,Pb,sg,API,Rs (scf/STB),x,Rs_Petrosky,Error %
0,710,2391.7,0.851,47.1,751,0.2008,772.0,2.80
1,680,2634.7,0.855,40.7,768,0.1566,726.0,5.47
2,720,2065.7,0.911,48.6,693,0.2101,757.0,9.24
3,697,2898.7,0.898,40.5,968,0.1457,834.0,13.84
4,678,3059.7,0.781,44.2,943,0.1900,865.0,8.27
5,640,4253.7,0.848,27.3,807,0.0667,900.0,11.52


In [ ]:
average_error_percentage = df5['Error %'].mean()
print(f" Absolute Average Error for Petrosky-Farshad: {average_error_percentage:.2f}%")

 Absolute Average Error for Petrosky-Farshad: 8.52%


From the experimental measured PVT data at the specified pressure and temperature

In [ ]:
def Rs_exp_pvt(Bo,sg,oil_density,sgo):
  Rs = round(((Bo*oil_density) - (62.4*sgo)) / (0.0136*sg),0)
  return Rs

In [ ]:
sg=0.851
API=47.1
Tsep=60+460
Psep=150+14.7
Pb=2377+14.7
T=250+460
Bo=1.528
oil_density=38.13
sgo = round(141.5 / (API + 131.5),4)
Rs = Rs_exp_pvt(Bo, sg, oil_density, sgo)

print(Rs)
print(sgo)

762.0
0.7923


In [ ]:
df6 = df[[ 'sg', 'API', 'Bo (bbl/STB)','ρo', 'Rs (scf/STB)']].copy()
df6["sgo"] = round(141.5 / (df6["API"] + 131.5),4)
df6["oil_density"] = df6["ρo"]
df6["Bo"] = df6["Bo (bbl/STB)"]
df7 = df6.drop(columns=['ρo', 'Bo (bbl/STB)'])
df7["Rs_exp_pvt"] = df7.apply(lambda row: round(Rs_exp_pvt(row['Bo'], row['sg'], row['oil_density'], row['sgo']), 0), axis=1)
df7["Error %"] = round(abs((df7["Rs (scf/STB)"] - df7["Rs_exp_pvt"])/df7["Rs (scf/STB)"]*100),2)
display(df7)

,sg,API,Rs (scf/STB),sgo,oil_density,Bo,Rs_exp_pvt,Error %
0,0.851,47.1,751,0.7923,38.13,1.528,762.0,1.46
1,0.855,40.7,768,0.8217,40.95,1.474,781.0,1.69
2,0.911,48.6,693,0.7857,37.37,1.529,655.0,5.48
3,0.898,40.5,968,0.8227,38.92,1.619,956.0,1.24
4,0.781,44.2,943,0.8054,37.70,1.570,841.0,10.82
5,0.848,27.3,807,0.8911,46.79,1.385,798.0,1.12


**Comparison between all correlation**

In [ ]:
dfc = df.copy()
dfc["sgo"] = round(141.5 / (df6["API"] + 131.5),4)
dfc["oil_density"] = df6["ρo"]
dfc["Bo"] = df6["Bo (bbl/STB)"]
dfc1 = dfc.drop(columns=['ρo', 'Bo (bbl/STB)', 'co (1/psi)', 'P (psig)'])
dfc1["Rs_standing"] = dfc1.apply(lambda row: standing_rs(row['Pb'], row['T'], row['API'], row['sg']), axis=1)
dfc1["Rs_Vasquez"] = dfc1.apply(lambda row: Rs_vasquez(row['sg'], row['API'], row['Tsep'], row['Psep'], row['Pb'], row['T']), axis=1)
dfc1["Rs_Glaso"] = dfc1.apply(lambda row: Rs_glaso(row['Pb'], row['T'], row['API'], row['sg']), axis=1)
dfc1["Rs_Marhoun"] = dfc1.apply(lambda row: Rs_marhoun(row['sg'], row['T'], row['Pb'], row['sgo']), axis=1)
dfc1["Rs_Petrosky"] = dfc1.apply(lambda row: Rs_petrosky(row['sg'], row['API'], row['Pb'], row['T']), axis=1)
dfc1["Rs_exp_pvt"] = dfc1.apply(lambda row: Rs_exp_pvt(row['Bo'], row['sg'], row['oil_density'], row['sgo']), axis=1)
dfc2 = dfc1[["Rs (scf/STB)", "Rs_standing", "Rs_Vasquez", "Rs_Glaso", "Rs_Marhoun", "Rs_Petrosky", "Rs_exp_pvt"]]
dfc2

,Rs (scf/STB),Rs_standing,Rs_Vasquez,Rs_Glaso,Rs_Marhoun,Rs_Petrosky,Rs_exp_pvt
0,751,837.0,779.0,737.0,740.0,772.0,762.0
1,768,817.0,733.0,714.0,792.0,726.0,781.0
2,693,774.0,702.0,686.0,729.0,757.0,655.0
3,968,915.0,782.0,826.0,978.0,834.0,956.0
4,943,1011.0,947.0,867.0,845.0,865.0,841.0
5,807,997.0,841.0,842.0,1186.0,900.0,798.0


**Bubble-Point Pressure**

The highest pressure at which a bubble of gas is first liberated from the oil.

• Standing

• Vasquez and Beggs

• Glaso

• Marhoun

• Petrosky and Farshad

**Standing’s Correlation**

In [ ]:
def Pb_standing(T, API, sg, Rs):
  a = (0.00091 * (T-460)) - (0.0125 * API)
  Pb = 18.2*((Rs/sg)**0.83 * 10**a - 1.4)
  return Pb

In [ ]:
T=250+460
API=47.1
sg=0.851
Rs=751
a = round((0.00091 * (T-460)) - (0.0125 * API),4)
Pb = round(Pb_standing(T, API, sg, Rs),0)
print(a)
print(Pb)

-0.3612
2181.0


**Example**

In [ ]:
import pandas as pd

data = {
    "Oil #": [1, 2, 3, 4, 5, 6],
    "T": [250+460, 220+460, 260+460, 237+460, 218+460, 180+460],
    "Pb": [2377+14.7, 2620+14.7, 2051+14.7, 2884+14.7, 3045+14.7, 4239+14.7],
    "Rs": [751, 768, 693, 968, 943, 807],
    "Bo": [1.528, 1.474, 1.529, 1.619, 1.570, 1.385],
    "oil_density": [38.13, 40.95, 37.37, 38.92, 37.70, 46.79],
    "co (1/psi)": [22.14e-6, 18.75e-6, 22.69e-6, 21.51e-6, 24.16e-6, 11.45e-6],
    "P (psig)": [2689, 2810, 2526, 2942, 3273, 4370],
    "Psep": [150+14.7, 100+14.7, 100+14.7, 60+14.7, 200+14.7, 85+14.7],
    "Tsep": [60+460, 75+460, 72+460, 120+460, 60+460, 173+460],
    "API": [47.1, 40.7, 48.6, 40.5, 44.2, 27.3],
    "sg": [0.851, 0.855, 0.911, 0.898, 0.781, 0.848]
}

df = pd.DataFrame(data)

df

,Oil #,T,Pb,Rs,Bo,oil_density,co (1/psi),P (psig),Psep,Tsep,API,sg
0,1,710,2391.7,751,1.528,38.13,0.000022,2689,164.7,520,47.1,0.851
1,2,680,2634.7,768,1.474,40.95,0.000019,2810,114.7,535,40.7,0.855
2,3,720,2065.7,693,1.529,37.37,0.000023,2526,114.7,532,48.6,0.911
3,4,697,2898.7,968,1.619,38.92,0.000022,2942,74.7,580,40.5,0.898
4,5,678,3059.7,943,1.570,37.70,0.000024,3273,214.7,520,44.2,0.781
5,6,640,4253.7,807,1.385,46.79,0.000011,4370,99.7,633,27.3,0.848


In [ ]:
df1 = df.loc[:, ['T', 'API', 'sg', 'Rs', 'Pb']].copy()
df1["a"] = round((0.00091 * (df1['T']-460)) - (0.0125 * df1['API']),4)
df1["Pb_Standing"] = df1.apply(lambda row: round(Pb_standing(row['T'], row['API'], row['sg'], row['Rs']),0), axis=1)
df1["Error %"] = round(abs((df1["Pb"] - df1["Pb_Standing"])/df1["Pb"]*100),2)
display(df1)

,T,API,sg,Rs,Pb,a,Pb_Standing,Error %
0,710,47.1,0.851,751,2391.7,-0.3612,2181.0,8.81
1,680,40.7,0.855,768,2634.7,-0.3086,2503.0,5.00
2,720,48.6,0.911,693,2065.7,-0.3709,1883.0,8.84
3,697,40.5,0.898,968,2898.7,-0.2906,3040.0,4.87
4,678,44.2,0.781,943,3059.7,-0.3541,2885.0,5.71
5,640,27.3,0.848,807,4253.7,-0.1775,3562.0,16.26


McCain (1991) suggested that by replacing the specific gravity of the gas in Equation 2-77 with that of the separator gas

In [ ]:
def Pb_vasquez(sg,API,Tsep,Psep,Rs,T):
  sg_s= sg*(1+5.912*(10**-5)*API*(Tsep-460)*log10(Psep/114.7))
  if API > 30:
    C1=56.18
    C2=0.84246
    C3=10.393
  else:
    C1=27.624
    C2=0.914328
    C3=11.172

  a = - C3 * (API/T)
  Pb = (((C1*Rs)/sg_s)*(10**a))**C2
  return Pb


In [ ]:
sg=0.851
API=47.1
Tsep=60+460
Psep=150+14.7
T=250+460
Bo=1.528
oil_density=38.13
sg_s= sg*(1+5.912*(10**-5)*API*(Tsep-460)*log10(Psep/114.7))
Rs=751 # Add Rs variable for correct function call
Pb=round(Pb_vasquez(sg,API,Tsep,Psep,Rs,T),0)
print(sg_s)
print(Pb)

0.8733406474157185
2319.0


In [ ]:
df2 = df.loc[:, ['T', 'API', 'sg', 'Rs', 'Tsep','Psep','Pb']].copy()
df2["Pb_Vasquez"] = df2.apply(lambda row: round(Pb_vasquez(row['sg'], row['API'], row['Tsep'], row['Psep'], row['Rs'], row['T']),0), axis=1)
df2["Error %"] = round(abs((df2["Pb"] - df2["Pb_Vasquez"])/df2["Pb"]*100),2)
display(df2)

,T,API,sg,Rs,Tsep,Psep,Pb,Pb_Vasquez,Error %
0,710,47.1,0.851,751,520,164.7,2391.7,2319.0,3.04
1,680,40.7,0.855,768,535,114.7,2634.7,2742.0,4.07
2,720,48.6,0.911,693,532,114.7,2065.7,2043.0,1.10
3,697,40.5,0.898,968,580,74.7,2898.7,3469.0,19.67
4,678,44.2,0.781,943,520,214.7,3059.7,3049.0,0.35
5,640,27.3,0.848,807,633,99.7,4253.7,4094.0,3.75


**Glaso’s Correlation**

In [ ]:
import math
def Pb_glaso(sg,API,Rs,t):
  a = 0.816
  c = -0.989
  b = 0.172 #if volatile oil = 0.130
  x = (Rs/sg)**a * ((t)**b) * (API)**c
  log_Pb = 1.7669 + 1.7447 * math.log10(x) - 0.30218 * ((math.log10(x)) ** 2)
  Pb = 10 ** log_Pb
  return Pb

In [ ]:
sg=0.851
API=47.1
Tsep=60+460
Psep=150+14.7
t=250
Bo=1.528
oil_density=38.13
Rs=751
a = 0.816
c = -0.989
b = 0.172
x = (Rs/sg)**a * ((t)**b) * (API)**c
log_Pb = 1.7669 + 1.7447 * math.log10(x) - 0.30218 * ((math.log10(x)) ** 2)
Pb = 10 ** log_Pb

print(x)
print(log_Pb)
print(Pb)

14.5053319487056
3.385732229953864
2430.704863509946


In [ ]:
df3 = df.loc[:, ['T', 'API', 'sg', 'Rs', 'Tsep','Psep','Pb']].copy()
df3["t"] = df3["T"] - 460
df3["x"] = round((df3["Rs"]/df3["sg"])**a * ((df3["t"])**b) * ((df3["API"])**c),4)
df3["Pb_Glaso"] = df3.apply(lambda row: round(Pb_glaso(row['sg'], row['API'], row['Rs'], row['t']),0), axis=1)
df3["Error %"] = round(abs((df3["Pb"] - df3["Pb_Glaso"])/df3["Pb"]*100),2)
display(df3)

,T,API,sg,Rs,Tsep,Psep,Pb,t,x,Pb_Glaso,Error %
0,710,47.1,0.851,751,520,164.7,2391.7,250,14.5053,2431.0,1.64
1,680,40.7,0.855,768,535,114.7,2634.7,220,16.6333,2797.0,6.16
2,720,48.6,0.911,693,532,114.7,2065.7,260,12.5419,2083.0,0.84
3,697,40.5,0.898,968,580,74.7,2898.7,237,19.6465,3295.0,13.67
4,678,44.2,0.781,943,520,214.7,3059.7,218,19.4845,3269.0,6.84
5,640,27.3,0.848,807,633,99.7,4253.7,180,25.0023,4125.0,3.03


**Marhoun’s Correlation**

In [ ]:
def Pb_marhoun(sg,API,T,Rs,sgo):
  a = 5.38088*(10**-3)
  b = 0.715082
  c = -1.87784
  d = 3.1437
  e = 1.32657
  Pb = a * (Rs**b) * (sg**c) * (sgo**d) * (T**e)
  return Pb

In [ ]:
df4 = df.loc[:, ['T', 'API', 'sg', 'Rs', 'Pb']].copy()
df4["sgo"] = round(141.5 / (df4["API"] + 131.5),4)
df4["Pb_Marhoun"] = df4.apply(lambda row: round(Pb_marhoun(row['sg'], row['API'], row['T'], row['Rs'], row['sgo']),0), axis=1)
df4["Error %"] =  round(abs((df4["Pb"] - df4["Pb_Marhoun"])/df4["Pb"]*100),2)
df4

,T,API,sg,Rs,Pb,sgo,Pb_Marhoun,Error %
0,710,47.1,0.851,751,2391.7,0.7923,2417.0,1.06
1,680,40.7,0.855,768,2634.7,0.8217,2578.0,2.15
2,720,48.6,0.911,693,2065.7,0.7857,1993.0,3.52
3,697,40.5,0.898,968,2898.7,0.8227,2878.0,0.71
4,678,44.2,0.781,943,3059.7,0.8054,3310.0,8.18
5,640,27.3,0.848,807,4253.7,0.8911,3230.0,24.07


**The Petrosky-Farshad Correlation**

In [ ]:
def Pb_petrosky(sg,API,T,Rs):
  x = round((7.916*(10**-4) * (API**1.5410)) - (4.561*(10**-5)*(T-460)**1.3911),4)
  y = (10**x)
  Pb = ((112.727*(Rs**0.577421))/((sg**0.8439)*y)) - 1391.051
  return Pb


In [ ]:
df5 = df.loc[:, ['T', 'API', 'sg', 'Rs', 'Pb']].copy()
df5["Pb_Petrosky"] = df5.apply(lambda row: round(Pb_petrosky(row['sg'], row['API'], row['T'], row['Rs']),0), axis=1)
df5["Error %"] =  round(abs((df5["Pb"] - df5["Pb_Petrosky"])/df5["Pb"]*100),2)
df5

,T,API,sg,Rs,Pb,Pb_Petrosky,Error %
0,710,47.1,0.851,751,2391.7,2331.0,2.54
1,680,40.7,0.855,768,2634.7,2767.0,5.02
2,720,48.6,0.911,693,2065.7,1893.0,8.36
3,697,40.5,0.898,968,2898.7,3285.0,13.33
4,678,44.2,0.781,943,3059.7,3288.0,7.46
5,640,27.3,0.848,807,4253.7,3909.0,8.10


**Oil Formation Volume Factor**

 The ratio of the volume of oil (plus the gas in solution) at the prevailing reservoir temperature and pressure to the volume of oil at standard conditions.

• Standing’s correlation

• The Vasquez-Beggs correlation

• Glaso’s correlation

• Marhoun’s correlation

• The Petrosky-Farshad correlation

• Other correlations

**Standing’s Correlation**

In [ ]:
def Bo_standing(T,sg, Rs, sgo):
  Bo = 0.9759 + 0.000120 * ((Rs * (sg/sgo)**0.5) + (1.25 * (T-460)))**1.2
  return Bo


**The Vasquez-Beggs Correlation**

In [ ]:
def Bo_vasquez(sg,API,Tsep,Psep,Rs,T):
  sg_s= sg*(1+5.912*(10**-5)*API*(Tsep-460)*log10(Psep/114.7))
  if API > 30:
    C1= 4.670*10**-4
    C2= 1.100*10**-5
    C3= 1.337*10**-9
  else:
    C1= 4.677*10**-4
    C2= 1.751*10**-5
    C3= -1.811*10**-8
  Bo = (1.0 + C1 * Rs + (T - 520) * (API / sg_s) * (C2 + C3 * Rs))
  return Bo


**Glaso’s Correlation**

In [ ]:
def Bo_glaso(sg,Rs,T,sgo):
  x = (Rs * (sg/sgo)**0.526) + 0.968*(T-460)
  A = -6.58511 + 2.91329*math.log10(x) - 0.27683*((math.log10(x)) ** 2)
  Bo = 1.0 + 10**A
  return Bo

**Marhoun’s Correlation**

In [ ]:
def Bo_marhoun(Rs,sg,sgo,T):
  a = 0.742390
  b = 0.323294
  c = -1.202040
  F = (Rs**a) * (sg**b) * (sgo**c)
  Bo = 0.497069 + (0.862963*10**-3*T) + (0.182594*10**-2*F) +( 0.318099*10**-5*F**2)
  return Bo


**The Petrosky-Farshad Correlation**

In [ ]:
def Bo_petrosky(Rs,sg,sgo,T):
  Bo = 1.0113 + 7.2046*(10**-5) * (Rs**0.3738 * (sg**0.2914/sgo**0.6265) + 0.24626*(T-460)**0.5371)**3.0936
  return Bo

**Material Balance Equation**

In [ ]:
def Bo_MBE(sgo,sg,Rs,oil_density):
  Bo = ((62.4*sgo) + (0.0136*Rs*sg)) / oil_density
  return Bo

**EXAMPLE**

In [ ]:
import pandas as pd

data = {
    "Oil #": [1, 2, 3, 4, 5, 6],
    "T": [250+460, 220+460, 260+460, 237+460, 218+460, 180+460],
    "Pb": [2377+14.7, 2620+14.7, 2051+14.7, 2884+14.7, 3045+14.7, 4239+14.7],
    "Rs": [751, 768, 693, 968, 943, 807],
    "Bo": [1.528, 1.474, 1.529, 1.619, 1.570, 1.385],
    "oil_density": [38.13, 40.95, 37.37, 38.92, 37.70, 46.79],
    "co": [22.14e-6, 18.75e-6, 22.69e-6, 21.51e-6, 24.16e-6, 11.45e-6],
    "P": [2689, 2810, 2526, 2942, 3273, 4370],
    "Psep": [150+14.7, 100+14.7, 100+14.7, 60+14.7, 200+14.7, 85+14.7],
    "Tsep": [60+460, 75+460, 72+460, 120+460, 60+460, 173+460],
    "API": [47.1, 40.7, 48.6, 40.5, 44.2, 27.3],
    "sg": [0.851, 0.855, 0.911, 0.898, 0.781, 0.848]
}

df = pd.DataFrame(data)

df

,Oil #,T,Pb,Rs,Bo,oil_density,co,P,Psep,Tsep,API,sg
0,1,710,2391.7,751,1.528,38.13,0.000022,2689,164.7,520,47.1,0.851
1,2,680,2634.7,768,1.474,40.95,0.000019,2810,114.7,535,40.7,0.855
2,3,720,2065.7,693,1.529,37.37,0.000023,2526,114.7,532,48.6,0.911
3,4,697,2898.7,968,1.619,38.92,0.000022,2942,74.7,580,40.5,0.898
4,5,678,3059.7,943,1.570,37.70,0.000024,3273,214.7,520,44.2,0.781
5,6,640,4253.7,807,1.385,46.79,0.000011,4370,99.7,633,27.3,0.848


In [ ]:
dfx = df.copy()
dfx["sgo"] = round(141.5 / (dfx["API"] + 131.5),4)
dfx["Bo_Standing"] = dfx.apply(lambda row: round(Bo_standing(row['T'], row['sg'], row['Rs'], row['sgo']),3), axis=1)
dfx["Bo_Vasquez"] = dfx.apply(lambda row: round(Bo_vasquez(row['sg'], row['API'], row['Tsep'], row['Psep'], row['Rs'], row['T']),3), axis=1)
dfx["Bo_Glaso"] = dfx.apply(lambda row: round(Bo_glaso(row['sg'], row['Rs'], row['T'], row['sgo']),3), axis=1)
dfx["Bo_Marhoun"] = dfx.apply(lambda row: round(Bo_marhoun(row['Rs'], row['sg'], row['sgo'], row['T']),3), axis=1)
dfx["Bo_Petrosky"] = dfx.apply(lambda row: round(Bo_petrosky(row['Rs'], row['sg'], row['sgo'], row['T']),3), axis=1)
dfx["Bo_MBE"] = dfx.apply(lambda row: round(Bo_MBE(row['sgo'], row['sg'], row['Rs'], row['oil_density']),3), axis=1)
dfx1 = dfx[["Bo", "Bo_Standing", "Bo_Vasquez", "Bo_Glaso", "Bo_Marhoun", "Bo_Petrosky", "Bo_MBE"]]
dfx1

,Bo,Bo_Standing,Bo_Vasquez,Bo_Glaso,Bo_Marhoun,Bo_Petrosky,Bo_MBE
0,1.528,1.506,1.474,1.473,1.516,1.552,1.525
1,1.474,1.487,1.450,1.459,1.477,1.508,1.470
2,1.529,1.495,1.451,1.461,1.511,1.556,1.542
3,1.619,1.635,1.556,1.601,1.594,1.657,1.623
4,1.570,1.571,1.546,1.541,1.554,1.584,1.599
5,1.385,1.461,1.389,1.438,1.414,1.433,1.387


**Isothermal Compressibility Coefficient of Crude Oil**

• The Vasquez-Beggs correlation

• The Petrosky-Farshad correlation

• McCain’s correlation

**The Vasquez-Beggs Correlation**

In [ ]:
def co_vasquez(P,API,Rs,sg,Tsep,Psep):
  #Rs = gas solubility at the bubble-point pressure
  #p = pressure above the bubble-point pressure, psia
  sg_s= sg*(1+5.912*(10**-5)*API*(Tsep-460)*log10(Psep/114.7))
  co = (-1433 + 5*Rs + 17.2*(T-460) - 1180*sg_s + 12.61*API) / (10**5 * P)
  return co


**The Petrosky-Farshad Correlation**

In [ ]:
def co_petrosky(P,API,Rs,sg,T):
  co = (1.705*10**-7) * (Rs**0.69357) * (sg**0.1885) * (API**0.3272) * (T-460)**0.6729 * (P**-0.5906)
  return co

In [ ]:
df = df.copy()
df["co_Vasquez"] = df.apply(lambda row: (co_vasquez(row['P'], row['API'], row['Rs'], row['sg'], row['Tsep'], row['Psep'])), axis=1)
df["co_Petrosky"] = df.apply(lambda row: (co_petrosky(row['P'], row['API'], row['Rs'], row['sg'], row['T'])), axis=1)
df_co = df[["co", "co_Vasquez", "co_Petrosky"]]
df_co["co"] = df_co["co"] * 10**6
df_co["co_Vasquez"] = df_co["co_Vasquez"] * 10**6
df_co["co_Petrosky"] = df_co["co_Petrosky"] * 10**6
df_co

/tmp/ipykernel_8235/2691542698.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_co["co"] = df_co["co"] * 10**6
/tmp/ipykernel_8235/2691542698.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_co["co_Vasquez"] = df_co["co_Vasquez"] * 10**6
/tmp/ipykernel_8235/2691542698.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

,co,co_Vasquez,co_Petrosky
0,22.14,23.002562,22.307127
1,18.75,22.104367,19.327227
2,22.69,23.237791,23.002062
3,21.51,24.523346,23.397940
4,24.16,21.932299,20.442011
5,11.45,14.330937,11.796800


**McCain and coauthors correlation**

In [ ]:
def co_mccain(P,API,Rs,sg,T):
  A = (
        -7.633
        - 1.497 * math.log(P)
        + 1.115 * math.log(T)
        + 0.533 * math.log(API)
        + 0.184 * math.log(Rs)
    )
  co = math.exp(A)
  return co

def co_mccain(P,API,Rs,sg,T,Pb):
   A = (
        -7.573
        - 1.45 * math.log(P)
        - 0.383 * math.log(Pb)
        + 1.402 * math.log(T)
        + 0.256 * math.log(API)
        + 0.449 * math.log(Rs)
    )
   co = math.exp(A)
   return co

In [ ]:
import math

def oil_compressibility(Rs, Bo, sg, sgo, T, Bg, P):
    term = (
        0.00014
        * math.sqrt(sg / sgo)
        * (
            Rs * math.sqrt(sg / sgo)
            + 1.25 * (T - 460)
        )**0.12 - Bg)

    co = (-Rs / (Bo * (0.83 * P + 21.75))) * term

    return co

**Example**

A crude oil system exists at 1650 psi and a temperature of 250°F. The system has the following PVT properties:

In [ ]:
#Input
API = 47.1
Pb = 2377
sg = 0.851
sg_s = 0.873
Rs = 751
Bo = 1.528
P= 1665
T=250+460

In [ ]:
A = round(-7.573 - 1.45 * math.log(P) - 0.383 * math.log(Pb) + 1.402 * math.log(T) + 0.256 * math.log(API) + 0.449 * math.log(Rs),4)
co = (co_mccain(P, API, Rs, sg, T, Pb)) / 10**-6
print(A)
print(f"{co} * 10^-6")

-8.1421
291.0301317540626 * 10^-6


In [ ]:
# lab pvt @1650psig
Bo = 1.393
Bg = 0.0001936
Rs = 515
co = 0.001936
P=1650+14.7
sgo = round(141.5 / (API + 131.5),3)
term = ((0.00014 * (sg / sgo)**0.5) * ((Rs * (sg / sgo)**0.5) + (1.25 * (T - 460)))**0.12 - Bg)
x = (-Rs / (Bo * (0.83 * P + 21.75)))
print(term)
print(sgo)
print(x)

0.00013226332214844977
0.792
-0.26342613401765286


**Oil Formation Volume Factor for Undersaturated Oils**

In [ ]:
# Bo = Bob * exp(-co *(p-pb)) - original

#replacing with the Vasquez-Beggs’ ,co expression

def Boi_vasquez(Rs,T,sg_s,API,P,Pb):
  #Rs = Rs at Pb
  #Bo = Bo at Pb
  A = 10**-5 * (-1433 + (5*Rs) + 17.2*(T - 460) - (1180*sg_s) + 12.61*API)
  Boi = round(Bo * exp(-A * math.log(P/Pb)),3)
  return Boi

def Boi_petrosky(Rs,sg,API,T,Bo,P,Pb):
  #Rs = Rs at Pb
  #Bo = Bo at Pb
  A = 4.1646*10**-7 * (Rs**0.69357) * (sg**0.1885) * (API**0.3272) * (T-460)**0.6729
  Boi = round(Bo * exp(-A * (P**0.4094 - Pb**0.4094)),3)
  return Boi

In [ ]:
API = 47.1
Pb = 2377
sg = 0.851
sg_s = 0.873
Rs = 751
Bo = 1.528
P= 5000+14.7
T=250+460
A = 10**-5 * (-1433 + (5*Rs) + 17.2*(T - 460) - (1180*sg_s) + 12.61*API)
Boi_1 = Boi_vasquez(Rs,T,sg_s,API,P,Pb)
Boi_2 = Boi_petrosky(Rs,sg,API,T,Bo,P,Pb)
print(A)
print(Boi_1)
print(Boi_2)

0.061857909999999995
1.459
1.454


**Crude Oil Density**

The mass of a unit volume of the crude at a specified pressure and temperature

In [ ]:
#Calculate the density of the oil at pressure below or equal to the bubble-point pressure

def oil_density_pb(sgo,Rs,sg,Bo):
  oil_d = ((62.4*sgo) + (0.0136*Rs*sg)) / Bo
  return oil_d

#Standing

def oil_density_standing(sgo,Rs,sg,T):
  oil_d = (((62.4*sgo) + (0.0136*Rs*sg))) / (0.972 + 0.000147*(Rs * (sg/sgo)**0.5 + 1.25 * (T-460))**1.175)
  return oil_d

#Density of the oil at pressures above the bubble-point pressure
# rho_oil = rho_oil_pb * exp(co/(P-Pb))
#Vasquez
def oil_density_vasquez(density_oil,Rs,P,Pb,API,T,sg_s):
  A = 10**-5 * (-1433 +( 5*Rs) + (17.2 *(T - 460)) - (1180 * sg_s) + (12.61* API))
  oil_d = density_oil * exp(A* math.log(P/Pb))
  return oil_d

#Petrosky
def oil_density_petrosky(density_oil,Rs,P,Pb,API,T,sg_s):
  A = 4.1646*10**-7 * (Rs**0.69357) * (sg**0.1885) * (API**0.3272) * (T-460)**0.6729
  oil_d = density_oil * exp(A * (P**0.4094 - Pb**0.4094))
  return oil_d



In [ ]:
df = df.copy()
df["sgo"] = round(141.5 / (df["API"] + 131.5),4)
df1 = df.loc[:, ['T','Bo' ,'API', 'sg', 'Rs', 'sgo','oil_density']]
df1["oil_density_below_pb"] = df1.apply(lambda row: round(oil_density_pb(row['sgo'], row['Rs'], row['sg'], row['Bo']),2), axis=1)
df1["oil_density_standing"] = df1.apply(lambda row: round(oil_density_standing(row['sgo'], row['Rs'], row['sg'], row['T']),2), axis=1)
df2 = df1[["oil_density", "oil_density_below_pb", "oil_density_standing"]]
df2

,oil_density,oil_density_below_pb,oil_density_standing
0,38.13,38.04,38.31
1,40.95,40.84,40.18
2,37.37,37.68,38.26
3,38.92,39.01,38.35
4,37.70,38.39,38.08
5,46.79,46.87,44.11


**Crude Oil Viscosity**

controls and influences the flow of oil through porous media and pipes

**• Dead-Oil Viscosity**

The dead-oil viscosity is defined as the viscosity of crude oil at atmospheric pressure (no gas in solution) and system temperature.

**• Saturated-Oil Viscosity**

The saturated (bubble-point)-oil viscosity is defined as the viscosity of the crude oil at the bubble-point pressure and reservoir temperature.

**• Undersaturated-Oil Viscosity**

The undersaturated-oil viscosity is defined as the viscosity of the crude oil at a pressure above the bubble-point and reservoir temperature.

**METHODS OF CALCULATING VISCOSITY
OF THE DEAD OIL**

• Beal’s correlation

• The Beggs-Robinson correlation

• Glaso’s correlation

In [ ]:
import math
def dead_oil_viscosity_beals(API,T):
  a = 10**(0.43+(8.33/API))
  u_od= (0.32+((1.8*10**7)/API**4.53)) * (360/(T-260))**a
  return u_od

def dead_oil_viscosity_beggs(API,T):
  Z = 3.0324 - (0.02023*API)
  Y = 10**Z
  x = Y*(T-460)**-1.163
  u_od = 10**x -1
  return u_od

def dead_oil_viscosity_glaso(API,T):
  a = (10.313 * (log10(T-460))) -36.447
  u_od = round((3.141*10**10)*((T-460)**-3.444) * (log10(API))**a,3)
  return u_od


In [ ]:
API=47.1
T=250+460
a = (10.313 * (log10(T-460))) -36.447
u_od = (3.141*10**10)*((T-460)**-3.444) * (log10(API))**a
print(a)
print(u_od)

-11.717044690565277
0.4166650054156487


In [ ]:
API=47.1
T=250+460
a = 10**(0.43+(8.33/API))
u_od= (0.32+((1.8*10**7)/API**4.53)) * (360/(T-260))**a
print(u_od)
print(a)

0.322327990910862
4.044433970112319


**METHODS OF CALCULATING THE
SATURATED OIL VISCOSITY**

• The Chew-Connally correlation

• The Beggs-Robinson correlation

In [ ]:
def sat_oil_viscosity_chew(Rs,u_od):
  e = (3.74*10**-3) * Rs
  d = (1.1*10**-3) * Rs
  c = (8.62*10**-5) * Rs
  b = (0.68/(10**c)) + (0.25/(10**d)) + (0.062/(10**e))
  a = Rs * ((2.2*10**-7 * Rs) - (7.4*10**-4))
  u_ob = (10**a) * (u_od**b)
  return u_ob

def sat_oil_viscosity_beggs(Rs,u_od):
  a = 10.715*(Rs+100)**-0.515
  b = 5.44*(Rs+150)**-0.338
  u_ob = a * (u_od**b)
  return u_ob



**METHODS OF CALCULATING THE
VISCOSITY OF THE UNDERSATURATED OIL**

The Vasquez-Beggs Correlation

In [ ]:
def undersat_oil_viscosity_vb(P,Pb,u_ob):
  a = (-3.9*10**-5*P)-5
  m = 2.6*(P**1.187)*10**a
  u_o = u_ob*(P/Pb)**m
  return u_o

**Example**

Using all the oil viscosity correlations discussed in this chapter, please calculate mod, mob, and the viscosity of the undersaturated oil.

In [ ]:
import pandas as pd

data = {
    "Oil #": [1, 2, 3, 4, 5, 6],
    "T": [250+460, 220+460, 260+460, 237+460, 218+460, 180+460],
    "Pb": [2377+14.7, 2620+14.7, 2051+14.7, 2884+14.7, 3045+14.7, 4239+14.7],
    "Rs": [751, 768, 693, 968, 943, 807],
    "Bo": [1.528, 1.474, 1.529, 1.619, 1.570, 1.385],
    "oil_density": [38.13, 40.95, 37.37, 38.92, 37.70, 46.79],
    "co": [22.14e-6, 18.75e-6, 22.69e-6, 21.51e-6, 24.16e-6, 11.45e-6],
    "P": [2689, 2810, 2526, 2942, 3273, 4370],
    "Psep": [150+14.7, 100+14.7, 100+14.7, 60+14.7, 200+14.7, 85+14.7],
    "Tsep": [60+460, 75+460, 72+460, 120+460, 60+460, 173+460],
    "API": [47.1, 40.7, 48.6, 40.5, 44.2, 27.3],
    "sg": [0.851, 0.855, 0.911, 0.898, 0.781, 0.848]
}

df = pd.DataFrame(data)

df

,Oil #,T,Pb,Rs,Bo,oil_density,co,P,Psep,Tsep,API,sg
0,1,710,2391.7,751,1.528,38.13,0.000022,2689,164.7,520,47.1,0.851
1,2,680,2634.7,768,1.474,40.95,0.000019,2810,114.7,535,40.7,0.855
2,3,720,2065.7,693,1.529,37.37,0.000023,2526,114.7,532,48.6,0.911
3,4,697,2898.7,968,1.619,38.92,0.000022,2942,74.7,580,40.5,0.898
4,5,678,3059.7,943,1.570,37.70,0.000024,3273,214.7,520,44.2,0.781
5,6,640,4253.7,807,1.385,46.79,0.000011,4370,99.7,633,27.3,0.848


In [ ]:
dfz = df.loc[:,["API","T"]]
dfz["Dead_Oil_Viscosity_Beals"] = dfz.apply(lambda row: round(dead_oil_viscosity_beals(row["API"], row["T"]),3), axis=1)
dfz["Dead_Oil_Viscosity_Beggs"] = dfz.apply(lambda row: round(dead_oil_viscosity_beggs(row["API"], row["T"]),3), axis=1)
dfz["Dead_Oil_Viscosity_Glaso"] = dfz.apply(lambda row: round(dead_oil_viscosity_glaso(row["API"], row["T"]),3), axis=1)
dfz

,API,T,Dead_Oil_Viscosity_Beals,Dead_Oil_Viscosity_Beggs,Dead_Oil_Viscosity_Glaso
0,47.1,710,0.322,0.568,0.417
1,40.7,680,0.638,1.020,0.775
2,48.6,720,0.275,0.493,0.363
3,40.5,697,0.546,0.917,0.714
4,44.2,678,0.512,0.829,0.598
5,27.3,640,4.425,4.246,4.536


In [ ]:
dfx = df.loc[:,["Rs"]]
dfx["u_od"] = [0.765,0.286,0.686,1.014,1.009,4.166]
dfx["Sat_Oil_Viscosity_Chew"] = dfx.apply(lambda row: round(sat_oil_viscosity_chew(row["Rs"], row["u_od"]),3), axis=1)
dfx["Sat_Oil_Viscosity_Beggs"] =dfx.apply(lambda row: round(sat_oil_viscosity_beggs(row["Rs"], row["u_od"]),3), axis=1)
dfx

,Rs,u_od,Sat_Oil_Viscosity_Chew,Sat_Oil_Viscosity_Beggs
0,751,0.765,0.313,0.287
1,768,0.286,0.168,0.167
2,693,0.686,0.308,0.279
3,968,1.014,0.311,0.297
4,943,1.009,0.316,0.300
5,807,4.166,0.842,0.689


In [ ]:
dfy = df.loc[:,["Pb"]]
dfy["u_ob"] = [0.224,0.373,0.221,0.377,0.305,0.950]
dfy["P"] = [5000+14.7,5000+14.7,5000+14.7,6000+14.7,6000+14.7,5000+14.7]
dfy["Undersat_Oil_Viscosity_Vasquez"] = dfy.apply(lambda row: round(undersat_oil_viscosity_vb(row["P"], row["Pb"], row["u_ob"]),3), axis=1)
dfy

,Pb,u_ob,P,Undersat_Oil_Viscosity_Vasquez
0,2391.7,0.224,5014.7,0.303
1,2634.7,0.373,5014.7,0.485
2,2065.7,0.221,5014.7,0.318
3,2898.7,0.377,6014.7,0.529
4,3059.7,0.305,6014.7,0.417
5,4253.7,0.950,5014.7,1.016


In [ ]:
P=5000+14.7
Pb=2391.7
u_ob=0.224

u_o = undersat_oil_viscosity_vb(P,Pb,u_ob)
print(u_o)

0.3031915189129612


**Surface/Interfacial Tension**

In [ ]:
components = ["C1", "C2", "C3", "n-C4", "n-C5", "C6", "C7+"]
Mi= [16.043,30.070,44.097,58.123,72.150,86.177,215]
xi = [0.45, 0.05, 0.05, 0.03, 0.01, 0.01, 0.40]
yi = [0.77, 0.08, 0.06, 0.04, 0.02, 0.02, 0.01]

oil_density = 46.23
gas_density = 18.21

Ma_g =  sum(yi * Mi for yi, Mi in zip(yi, Mi))
Ma_o =  sum(xi * Mi for xi, Mi in zip(xi, Mi))
print(Ma_g)
print(Ma_o)

25.04599
100.25466


In [ ]:
import pandas as pd

data = {
    "Component": components,
    "Mi": Mi,
    "xi": xi,
    "yi": yi
}

dfc = pd.DataFrame(data)
display(dfc)

,Component,Mi,xi,yi
0,C1,16.043,0.45,0.77
1,C2,30.070,0.05,0.08
2,C3,44.097,0.05,0.06
3,n-C4,58.123,0.03,0.04
4,n-C5,72.150,0.01,0.02
5,C6,86.177,0.01,0.02
6,C7+,215.000,0.40,0.01


In [ ]:
A = round(oil_density/(62.4*Ma_o),5)
B = round(gas_density/(62.4*Ma_g),5)
print(A)
print(B)

0.00739
0.01165


In [ ]:
M7=215
pch_c7 = 69.9 + (2.3*M7)
print(pch_c7)

564.4


In [ ]:
dfc["pch"] = [77,108,150.3,189.9,231.5,271.0,564.4]
dfc["Axi"] = round(dfc["xi"] * A,5)
dfc["Byi"] = round(dfc["yi"] * B,5)
dfc["pch(Axi-Byi)"] = round(dfc["pch"] * (dfc["Axi"] - dfc["Byi"]),5)
dfc

,Component,Mi,xi,yi,pch,Axi,Byi,pch(Axi-Byi)
0,C1,16.043,0.45,0.77,77.0,0.00333,0.00897,-0.43428
1,C2,30.070,0.05,0.08,108.0,0.00037,0.00093,-0.06048
2,C3,44.097,0.05,0.06,150.3,0.00037,0.00070,-0.04960
3,n-C4,58.123,0.03,0.04,189.9,0.00022,0.00047,-0.04748
4,n-C5,72.150,0.01,0.02,231.5,0.00007,0.00023,-0.03704
5,C6,86.177,0.01,0.02,271.0,0.00007,0.00023,-0.04336
6,C7+,215.000,0.40,0.01,564.4,0.00296,0.00012,1.60290


In [ ]:
v = round(sum(dfc["pch(Axi-Byi)"]),3)
print(v)
surface_tension = round(v**4,3)
print(f"{surface_tension} dynes/cm")

0.931
0.751 dynes/cm


**PROPERTIES OF RESERVOIR WATER**

Water Formation Volume Factor

In [ ]:
def Bw(T, P, gas_free):
  if gas_free == True:
    A1 = (0.9947) + (5.8*(10**-6))*(T - 460) + (1.02*(10**-6))*(T - 460)**2
    A2 = (-4.228*(10**-6)) + (1.8376*(10**-8))*(T - 460) + (-6.77*(10**-11))*(T - 460)**2
    A3 = (1.3*(10**-10)) + (-1.3855*(10**-12))*(T - 460) + (4.285*(10**-15))*(T - 460)**2
  else: #Gas-Free water
    A1 = (0.9911) + (6.35*(10**-5))*(T - 460) + (8.5*(10**-7))*(T - 460)**2
    A2 = (-1.093*(10**-6)) + (-3.497*(10**-9))*(T - 460) + (4.57*(10**-12))*(T - 460)**2
    A3 = (-5*(10**-11)) + (6.429*(10**-13))*(T - 460) + (-1.43*(10**-15))*(T - 460)**2

  Bw = A1 + A2*P + A3*P**2
  return Bw

Water Viscosity

In [ ]:
# Y= water salinity, ppm
def water_viscosity(T,Y,P):
  A = (4.518*10**-2) + (9.313*10**-7*Y) - (3.93*10**-12*Y**2)
  B = (70.634 + 9.576*10**-10*Y**2)
  uwD = A + (B/T) #uwD = brine viscosity at p = 14.7, T, cp
  uw = uwD * (1+(3.5*10**-2*P**2*(T-40)))
  return uw # in cp

Gas Solubility in Water

In [ ]:
def Rsw(T,P):
  A = 2.12 + (3.45*(10**-3)*T) - (3.59*(10**-5)*T**2)
  B = 0.0107 - (5.26*(10**-5)*T) +( 1.48*(10**-7)*T**2)
  C = (8.75*(10**-7)) + (3.9*(10**-9)*T) - (1.02*(10**-11)*T**2)
  Rsw = A + B*P + C*P**2
  return Rsw

Water Isothermal Compressibility

In [ ]:
def cw():
  C1 = 3.8546 - (0.000134*P)
  C2 = -0.01052 + (4.77*10**-7*P)
  C3 = (3.9267*10**-5) - (8.8*10**-10*P)
  cw = (C1 + C2*T + C3*T**2) * 10**-6
  return cw
